# Task 3 — Reranking for Beyond-Accuracy

## Task 3.1 — Rerankers

Task 2 evaluates the lists the models already return. A reranker starts from those scores and builds a new top-10, with an explicit weight on a beyond-accuracy property the original ranking does not optimise. Four rerankers are implemented here.

* **Diversification** — maximal marginal relevance, so the genres inside one list repeat less.
* **Calibration** — Steck's greedy calibration, so the genre mix of the list stays close to the user's own training history.
* **Item-side fairness** — the same greedy form, with a target that spreads slots across the head, the mid catalogue and the tail.
* **User-side fairness** — the gender gap in NDCG. The group that is ahead on the validation split gives up some of that lead. The other group keeps the model's own list.

The candidate pool is the model's top 100. The weight λ is chosen on the validation split and then frozen for the test lists.

### How to run

Run Task 1 and Task 2 first, or provide their saved artefacts,
then execute this notebook from top to bottom in the `recsys` kernel. The notebook reads the score matrices and the splits written by `Task1.ipynb`, and it checks the base lists against `runs/tables/table_2_2_selected.csv`. Nothing is trained. Tables and the figure go to `runs/tables/`. The reranked test lists go to `runs/rerank/`.

In [ ]:
# =============================================================================
# 3.1.0  Paths, side information, score matrices
# =============================================================================
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)

PROJECT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "RecBole_DSAIT4335").is_dir())
RUNS = PROJECT / "runs"
DATA = PROJECT / "RecBole_DSAIT4335" / "dataset" / "ml-100k"
K = 10
POOL = 100
BUDGET = 0.02
GRID = [0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.7, 0.9]
ALPHA_GENRE = 0.01
ALPHA_ITEM = 0.1
TARGET_ITEM = np.full(3, 1.0 / 3.0)

MODELS = [
    ("tuned-EASE", "EASE"),
    ("tuned-SLIMElastic", "SLIMElastic"),
    ("tuned-NGCF", "NGCF"),
    ("tuned-NeuMF", "NeuMF"),
    ("tuned-BPR", "BPR"),
]


def ground_truth(split):
    frame = pd.read_csv(RUNS / "splits" / f"ml-100k.{split}.tsv", sep="\t", dtype=str)
    truth = {}
    for user, item in zip(frame.iloc[:, 0], frame.iloc[:, 1]):
        truth.setdefault(user, set()).add(item)
    return truth


GT_TEST = ground_truth("test")
GT_VALID = ground_truth("valid")

_train = pd.read_csv(RUNS / "splits" / "ml-100k.train.tsv", sep="\t", dtype=str)
_counts = _train.iloc[:, 1].value_counts()
POP_PROB = (_counts / int(_counts.sum())).to_dict()
_order = sorted(POP_PROB, key=POP_PROB.get, reverse=True)
_head_n = max(1, int(round(0.1 * len(POP_PROB))))
_mid_n = int(round(0.30 * len(POP_PROB)))
HEAD = set(_order[:_head_n])
MID = set(_order[_head_n:_mid_n])
TAIL = set(_order[_mid_n:])

_items = pd.read_csv(DATA / "ml-100k.item", sep="\t", dtype=str)
_idc = [c for c in _items.columns if c.startswith("item_id")][0]
_clc = [c for c in _items.columns if c.startswith("class")][0]
GENRES = {
    str(i): frozenset(str(c).split()) if isinstance(c, str) and c else frozenset()
    for i, c in zip(_items[_idc], _items[_clc])
}
CATALOG = set(GENRES)
GENRE_NAMES = sorted({g for gs in GENRES.values() for g in gs})
GENRE_INDEX = {g: i for i, g in enumerate(GENRE_NAMES)}

_users = pd.read_csv(DATA / "ml-100k.user", sep="\t", dtype=str)
_ucol = [c for c in _users.columns if c.startswith("user_id")][0]
_gcol = [c for c in _users.columns if c.startswith("gender")][0]
GENDER = dict(zip(_users[_ucol].astype(str), _users[_gcol].astype(str)))


def load_scores(run_id):
    with np.load(RUNS / "scores" / f"{run_id}.npz") as npz:
        return {
            "test": npz["scores_test"].astype(np.float64),
            "valid": npz["scores_valid"].astype(np.float64),
            "users_test": np.asarray(npz["users_test"], dtype=str),
            "users_valid": np.asarray(npz["users_valid"], dtype=str),
            "items": np.asarray(npz["items"], dtype=str),
        }


SCORES = {run_id: load_scores(run_id) for run_id, _ in MODELS}
ITEMS = SCORES["tuned-EASE"]["items"]
USERS_TEST = SCORES["tuned-EASE"]["users_test"]
USERS_VALID = SCORES["tuned-EASE"]["users_valid"]
for _run, _pack in SCORES.items():
    assert np.array_equal(_pack["items"], ITEMS), _run
    assert np.array_equal(_pack["users_test"], USERS_TEST), _run
    assert np.array_equal(_pack["users_valid"], USERS_VALID), _run

ITEM_COL = {tok: j for j, tok in enumerate(ITEMS)}
GENRE_BIN = np.zeros((len(ITEMS), len(GENRE_NAMES)), dtype=np.float64)
for _j, _tok in enumerate(ITEMS):
    for _g in GENRES.get(_tok, ()):
        GENRE_BIN[_j, GENRE_INDEX[_g]] = 1.0
_size = GENRE_BIN.sum(1, keepdims=True)
_size[_size == 0] = 1.0
GENRE_DIST = GENRE_BIN / _size

_inter = GENRE_BIN @ GENRE_BIN.T
_card = GENRE_BIN.sum(1)
_union = _card[:, None] + _card[None, :] - _inter
SIM = np.divide(_inter, _union, out=np.zeros_like(_inter), where=_union > 0)
SIM[(_card[:, None] == 0) & (_card[None, :] == 0)] = 1.0

GROUP_INDEX = np.full(len(ITEMS), 2, dtype=int)
for _j, _tok in enumerate(ITEMS):
    if _tok in HEAD:
        GROUP_INDEX[_j] = 0
    elif _tok in MID:
        GROUP_INDEX[_j] = 1
GROUP_DIST = np.eye(3, dtype=np.float64)[GROUP_INDEX]

_train_items = {}
for _u, _i in zip(_train.iloc[:, 0], _train.iloc[:, 1]):
    _train_items.setdefault(_u, []).append(_i)


def _smoothed(counts, alpha):
    z = counts + alpha
    return z / z.sum()


PROFILE = {}
for _u, _its in _train_items.items():
    cols = [ITEM_COL[i] for i in _its if i in ITEM_COL]
    counts = GENRE_DIST[cols].sum(0) if cols else np.zeros(len(GENRE_NAMES))
    PROFILE[_u] = _smoothed(counts, ALPHA_GENRE)
PROFILE_TEST = np.stack([PROFILE[u] for u in USERS_TEST])
PROFILE_VALID = np.stack([PROFILE[u] for u in USERS_VALID])

print(
    f"catalog {len(CATALOG)} items | genres {len(GENRE_NAMES)} | "
    f"head {_head_n} / mid {len(MID)} / tail {len(TAIL)} | "
    f"{len(USERS_TEST)} users | pool {POOL}"
)

### The four rerankers

Every method sees the same candidates: the model's top 100, scored by that model's relevance after a per-user min–max over the legal candidates. That is the scaling used for the weighted hybrid in Task 1.3. Items the user has already seen are masked in the exported matrices. The list is grown greedily to length 10. λ is the weight on the beyond-accuracy term. At λ = 0 every method returns the original top-10, because the penalty term is then ignored and the pool is already in score order.

**Diversification (MMR).** Maximal marginal relevance (Carbonell and Goldstein, 1998). The next item maximises

\[
(1-\lambda)\, r(u,i) - \lambda \max_{j \in L} \mathrm{sim}(i,j)
\]

where \(\mathrm{sim}\) is the Jaccard similarity of the two genre sets, the complement of the distance behind ILD in Task 2.1. The first pick is still the model's top item: on an empty list the similarity penalty is zero. Later picks pay for sharing genres with what is already there.

**Calibration (Steck, 2018).** The target \(p_u\) is the genre distribution of the user's training history. Each training movie spreads its mass uniformly over the genres it carries. \(q\) is the same average over the movies already chosen. The next item maximises

\[
(1-\lambda)\, r(u,i) - \lambda\, \mathrm{KL}\big(p_u \,\|\, q(L \cup \{i\})\big)
\]

with the divergence in bits. Both distributions are additively smoothed with \(\alpha = 0.01\), so a genre that has not appeared yet does not make the KL infinite. Validation and test interactions are not part of \(p_u\).

**Item-side fairness.** The same update, over popularity instead of genre, and with one target for every user. A movie is head, mid or tail under the cuts from Task 2.5: the most popular 10% of training items, the next 20%, and the rest. A movie with no training interactions is counted as tail. The target is uniform, \((1/3, 1/3, 1/3)\), so the three bands get the same number of slots. The base lists do not look like that: EASE puts about 88% of its top-10 on the head. Smoothing is \(\alpha = 0.1\). The pool still binds the method. A tail movie can be promoted only when the model already placed it inside the top 100.

**User-side fairness.** The groups are the genders in `ml-100k.user`. On the validation split one gender has the higher mean NDCG under the base list. A reranker that sees scores, and not the held-out labels, has no way to raise the lower of those two means: the base top-10 is already the list that maximises the model's own scores. The gap is closed from the side that is ahead. Users in the lower group keep the base list. Users in the higher group receive an MMR list, and λ is the grid point that makes \(|\mathrm{NDCG}_F - \mathrm{NDCG}_M|\) smallest. Which group is higher is read from that model's base validation NDCG and then kept for the test lists.

### Choosing λ

The grid is \(0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50, 0.70, 0.90\). For each model and each method the value is chosen on validation, inside a fixed accuracy budget: validation NDCG may fall by at most 0.02 from that model's own base list. Inside the budget, diversification takes the highest ILD, calibration the lowest genre KL, item-side fairness the lowest exposure KL, and user-side fairness the smallest gender gap. A tie goes to the smaller λ. If nothing on the grid improves the target inside the budget, the base list is kept.

Relevance and Jaccard similarity both lie in \([0, 1]\). The two KL terms are in bits, so the same numeric λ is a different trade-off for the calibration and item-fairness rerankers than it is for MMR. The grid is searched separately for each method, which is what makes the selected points comparable.

In [ ]:
# =============================================================================
# 3.1.1  Rerankers and the metrics they are judged on
# =============================================================================
def candidate_pool(scores):
    """Per-user min-max relevance, and the top-POOL item columns in score order."""
    finite = np.isfinite(scores)
    masked = np.where(finite, scores, np.nan)
    low = np.nanmin(masked, axis=1, keepdims=True)
    high = np.nanmax(masked, axis=1, keepdims=True)
    span = np.where(high > low, high - low, 1.0)
    relevance = (masked - low) / span
    filled = np.where(finite, relevance, -np.inf)
    pool = np.argsort(-filled, axis=1, kind="stable")[:, :POOL]
    return relevance, pool


def mmr_lists(pool, relevance, lam):
    """Greedy MMR. lam = 0 returns the pool order, which is the base top-k."""
    if lam == 0:
        return pool[:, :K].copy()
    out = np.empty((pool.shape[0], K), dtype=int)
    for u in range(pool.shape[0]):
        cands = pool[u]
        rel = relevance[u, cands]
        local = SIM[np.ix_(cands, cands)]
        max_sim = np.zeros(POOL)
        alive = np.ones(POOL, dtype=bool)
        for t in range(K):
            score = (1.0 - lam) * rel - lam * max_sim
            score[~alive] = -np.inf
            j = int(np.argmax(score))
            out[u, t] = cands[j]
            alive[j] = False
            max_sim = np.maximum(max_sim, local[j])
    return out


def kl_lists(pool, relevance, features, target, lam, alpha):
    """Greedy relevance-vs-KL. `target` is (D,) or one row per user."""
    if lam == 0:
        return pool[:, :K].copy()
    n_users, dim = pool.shape[0], features.shape[1]
    shared = target.ndim == 1
    out = np.empty((n_users, K), dtype=int)
    for u in range(n_users):
        cands = pool[u]
        rel = relevance[u, cands]
        feats = features[cands]
        desired = target if shared else target[u]
        counts = np.zeros(dim)
        alive = np.ones(POOL, dtype=bool)
        for t in range(K):
            proposed = counts + feats
            q = (proposed + alpha) / (proposed.sum(1, keepdims=True) + alpha * dim)
            kl = np.sum(desired * np.log2(desired / q), axis=1)
            score = (1.0 - lam) * rel - lam * kl
            score[~alive] = -np.inf
            j = int(np.argmax(score))
            out[u, t] = cands[j]
            alive[j] = False
            counts = counts + feats[j]
    return out


def user_fair_lists(base, diversified, advantaged):
    """Keep the base list for the lower group. MMR list for the group ahead."""
    out = base.copy()
    out[advantaged] = diversified[advantaged]
    return out


_DISC = 1.0 / np.log2(np.arange(1, K + 1) + 1.0)


def evaluate(index_lists, users, truth, profiles):
    """Accuracy, ILD, novelty, coverage, genre KL, popularity-band KL, gender NDCG."""
    n = len(users)
    ndcg = np.zeros(n)
    recall = np.zeros(n)
    ild = np.zeros(n)
    novelty = np.zeros(n)
    head = np.zeros(n)
    mid = np.zeros(n)
    tail = np.zeros(n)
    cal_kl = np.zeros(n)
    item_kl = np.zeros(n)
    shown = set()
    dim = len(GENRE_NAMES)
    for i, user in enumerate(users):
        cols = index_lists[i]
        rec = [ITEMS[j] for j in cols]
        shown.update(rec)
        relevant = truth[user]
        hits = np.fromiter((1.0 if it in relevant else 0.0 for it in rec), float, K)
        ideal = _DISC[:min(len(relevant), K)].sum()
        ndcg[i] = float((hits * _DISC).sum() / ideal) if ideal else 0.0
        recall[i] = float(hits.sum() / len(relevant))
        genres = [GENRES.get(it, frozenset()) for it in rec]
        dists = []
        for a in range(K):
            for b in range(a + 1, K):
                union = len(genres[a] | genres[b])
                dists.append(1.0 - len(genres[a] & genres[b]) / union if union else 0.0)
        ild[i] = float(np.mean(dists))
        surprisal = [-np.log2(POP_PROB[it]) for it in rec if POP_PROB.get(it, 0) > 0]
        novelty[i] = float(np.mean(surprisal)) if surprisal else 0.0
        bands = GROUP_INDEX[cols]
        head[i] = float(np.mean(bands == 0))
        mid[i] = float(np.mean(bands == 1))
        tail[i] = float(np.mean(bands == 2))
        q_genre = _smoothed(GENRE_DIST[cols].sum(0), ALPHA_GENRE)
        cal_kl[i] = float(np.sum(profiles[i] * np.log2(profiles[i] / q_genre)))
        q_item = _smoothed(np.bincount(bands, minlength=3).astype(float), ALPHA_ITEM)
        item_kl[i] = float(np.sum(TARGET_ITEM * np.log2(TARGET_ITEM / q_item)))
    gender = np.array([GENDER[u] for u in users])
    ndcg_f = float(ndcg[gender == "F"].mean())
    ndcg_m = float(ndcg[gender == "M"].mean())
    return {
        "ndcg": float(ndcg.mean()),
        "recall": float(recall.mean()),
        "ild": float(ild.mean()),
        "novelty": float(novelty.mean()),
        "coverage": len(shown & CATALOG) / len(CATALOG),
        "cal_kl": float(cal_kl.mean()),
        "item_kl": float(item_kl.mean()),
        "head": float(head.mean()),
        "mid": float(mid.mean()),
        "tail": float(tail.mean()),
        "ndcg_F": ndcg_f,
        "ndcg_M": ndcg_m,
        "gap": abs(ndcg_f - ndcg_m),
        "_ndcg": ndcg,
        "_gender": gender,
    }

In [ ]:
# =============================================================================
# 3.1.2  Select λ on validation, rerank the test lists, write tables
# =============================================================================
METHOD_SPEC = [
    # name, column optimised, True if higher is better
    ("mmr", "ild", True),
    ("calibration", "cal_kl", False),
    ("item_fair", "item_kl", False),
    ("user_fair", "gap", False),
]


def select_lambda(records, metric, higher, base_value):
    """Best grid point inside the NDCG budget. None keeps the base list."""
    feasible = [r for r in records if r["ndcg"] >= r["base_ndcg"] - BUDGET]
    pool = feasible if feasible else [min(records, key=lambda r: r["lam"])]
    if higher:
        best = max(pool, key=lambda r: (r[metric], -r["lam"]))
        improved = best[metric] > base_value + 1e-12
    else:
        best = min(pool, key=lambda r: (r[metric], r["lam"]))
        improved = best[metric] < base_value - 1e-12
    return best if improved else None


def dump_lists(path, run_id, label, method, lam, users, raw, index_lists):
    recommendations = {}
    for i, user in enumerate(users):
        cols = index_lists[i]
        recommendations[str(user)] = {
            "items": [str(ITEMS[j]) for j in cols],
            "scores": [float(raw[i, j]) for j in cols],
        }
    payload = {
        "run_id": f"{label}__{method}",
        "base_run": run_id,
        "method": method,
        "lam": lam,
        "k": K,
        "pool": POOL,
        "recommendations": recommendations,
    }
    path.write_text(json.dumps(payload))


curve_rows = []
test_rows = []
rerank_dir = RUNS / "rerank"
rerank_dir.mkdir(exist_ok=True)
(RUNS / "tables").mkdir(exist_ok=True)

published = pd.read_csv(RUNS / "tables" / "table_2_2_selected.csv").set_index("label")

for run_id, label in MODELS:
    pack = SCORES[run_id]
    rel_v, pool_v = candidate_pool(pack["valid"])
    rel_t, pool_t = candidate_pool(pack["test"])
    base_v_idx = pool_v[:, :K]
    base_t_idx = pool_t[:, :K]
    base_v = evaluate(base_v_idx, USERS_VALID, GT_VALID, PROFILE_VALID)
    base_t = evaluate(base_t_idx, USERS_TEST, GT_TEST, PROFILE_TEST)
    for metric in ("ndcg@10", "ild@10"):
        key = "ndcg" if metric.startswith("ndcg") else "ild"
        gap = abs(base_t[key] - float(published.loc[label, metric]))
        assert gap < 1e-6, (label, metric, base_t[key], gap)

    adv_name = "M" if base_v["ndcg_M"] >= base_v["ndcg_F"] else "F"
    adv_mask_v = base_v["_gender"] == adv_name
    print(
        f"{label}: validation NDCG F {base_v['ndcg_F']:.4f} / M {base_v['ndcg_M']:.4f} "
        f"-> group ahead is {adv_name}"
    )

    built_v = {"base": base_v_idx}
    for lam in GRID:
        built_v[("mmr", lam)] = mmr_lists(pool_v, rel_v, lam)
        built_v[("calibration", lam)] = kl_lists(
            pool_v, rel_v, GENRE_DIST, PROFILE_VALID, lam, ALPHA_GENRE)
        built_v[("item_fair", lam)] = kl_lists(
            pool_v, rel_v, GROUP_DIST, TARGET_ITEM, lam, ALPHA_ITEM)
        built_v[("user_fair", lam)] = user_fair_lists(
            base_v_idx, built_v[("mmr", lam)], adv_mask_v)

    chosen = {}
    for name, metric, higher in METHOD_SPEC:
        records = []
        for lam in GRID:
            metrics = evaluate(built_v[(name, lam)], USERS_VALID, GT_VALID, PROFILE_VALID)
            row = {"label": label, "method": name, "lam": lam, "base_ndcg": base_v["ndcg"]}
            row.update({k: v for k, v in metrics.items() if not k.startswith("_")})
            records.append(row)
            curve_rows.append(row)
        zero = {"label": label, "method": name, "lam": 0.0, "base_ndcg": base_v["ndcg"]}
        zero.update({k: v for k, v in base_v.items() if not k.startswith("_")})
        curve_rows.append(zero)
        best = select_lambda(records, metric, higher, base_v[metric])
        chosen[name] = 0.0 if best is None else best["lam"]
        moved = "kept" if best is None else f"λ={best['lam']:.2f}"
        print(f"  {name:<12} {moved}")

    gender_t = np.array([GENDER[u] for u in USERS_TEST])
    adv_mask_t = gender_t == adv_name
    test_lists = {
        "base": base_t_idx,
        "mmr": mmr_lists(pool_t, rel_t, chosen["mmr"]),
        "calibration": kl_lists(pool_t, rel_t, GENRE_DIST, PROFILE_TEST, chosen["calibration"], ALPHA_GENRE),
        "item_fair": kl_lists(pool_t, rel_t, GROUP_DIST, TARGET_ITEM, chosen["item_fair"], ALPHA_ITEM),
        "user_fair": user_fair_lists(
            base_t_idx, mmr_lists(pool_t, rel_t, chosen["user_fair"]), adv_mask_t),
    }
    for name, _metric, _higher in [("base", None, None), *METHOD_SPEC]:
        lam = 0.0 if name == "base" else chosen[name]
        metrics = base_t if name == "base" else evaluate(
            test_lists[name], USERS_TEST, GT_TEST, PROFILE_TEST)
        n_changed = int(np.any(test_lists[name] != base_t_idx, axis=1).sum())
        row = {
            "label": label, "method": name, "lam": lam, "adv_group": adv_name,
            "lists_changed": n_changed,
        }
        row.update({k: v for k, v in metrics.items() if not k.startswith("_")})
        test_rows.append(row)
        if name != "base":
            dump_lists(
                rerank_dir / f"{label}__{name}_top10.json",
                run_id, label, name, lam, USERS_TEST, pack["test"], test_lists[name],
            )

CURVES = pd.DataFrame(curve_rows)
TEST = pd.DataFrame(test_rows)
CURVES.to_csv(RUNS / "tables" / "table_3_1_validation.csv", index=False)
TEST.to_csv(RUNS / "tables" / "table_3_1_test.csv", index=False)

_order = ["base", "mmr", "calibration", "item_fair", "user_fair"]
TEST["method"] = pd.Categorical(TEST["method"], _order, ordered=True)
TEST = TEST.sort_values(["label", "method"]).reset_index(drop=True)

_acc_cols = ["label", "method", "lam", "ndcg", "recall", "ild", "novelty", "coverage", "lists_changed"]
_fair_cols = ["label", "method", "lam", "cal_kl", "item_kl", "head", "mid", "tail", "ndcg_F", "ndcg_M", "gap"]
with pd.option_context("display.float_format", "{:.4f}".format):
    print("\ntest accuracy and diversity")
    print(TEST[_acc_cols].to_string(index=False))
    print("\ntest calibration and fairness")
    print(TEST[_fair_cols].to_string(index=False))

# Validation trade-off for EASE, the reference accurate model. The vertical line is the λ that was kept.
fig, axes = plt.subplots(2, 2, figsize=(11, 7.2))
_ylabels = {
    "mmr": "ILD@10",
    "calibration": "genre KL (bits)",
    "item_fair": "exposure KL (bits)",
    "user_fair": "|NDCG gap|",
}
_titles = {
    "mmr": "Diversification",
    "calibration": "Calibration",
    "item_fair": "Item-side fairness",
    "user_fair": "User-side fairness",
}
ease_lam = {r["method"]: r["lam"] for _, r in TEST[TEST["label"] == "EASE"].iterrows()}
for ax, (name, _metric, _higher) in zip(axes.ravel(), METHOD_SPEC):
    sub = CURVES[(CURVES["label"] == "EASE") & (CURVES["method"] == name)].sort_values("lam")
    line = ax.plot(sub["lam"], sub[_metric], color="#e6550d", marker="o", label=_ylabels[name])
    ax.set_xlabel("λ")
    ax.set_ylabel(_ylabels[name], color="#e6550d")
    twin = ax.twinx()
    twin.plot(sub["lam"], sub["ndcg"], color="#08519c", marker="s", label="NDCG@10")
    twin.set_ylabel("NDCG@10", color="#08519c")
    ax.axvline(ease_lam[name], color="#333333", ls="--", lw=1)
    ax.set_title(_titles[name])
    ax.grid(alpha=0.3)
fig.suptitle("Task 3.1 — validation trade-off on EASE (dashed line is the λ used at test)")
fig.tight_layout()
fig.savefig(RUNS / "tables" / "fig_task31.png", dpi=130, bbox_inches="tight")
plt.show()
print("saved runs/tables/table_3_1_validation.csv, table_3_1_test.csv, fig_task31.png")
print(f"saved {len(list(rerank_dir.glob('*_top10.json')))} reranked lists in runs/rerank/")

### Reading Task 3.1

λ is chosen on validation, inside a drop of 0.02 NDCG. EASE and SLIMElastic take 0.3 for MMR, 0.3 for calibration, 0.5 for item-side fairness and 0.4 for user-side fairness. NGCF takes 0.2. NeuMF and BPR take smaller values, because their validation NDCG is already lower and the same budget is spent sooner. On the test split the drop can run past 0.02: EASE item-side fairness goes from 0.330 to 0.303.

MMR rewrites all 943 lists and is what moves ILD off the flat band from Task 2.1. EASE goes from 0.810 to 0.883, and the other four land between 0.875 and 0.885. The cost on EASE is NDCG from 0.330 to 0.313. Calibration cuts the genre KL by about half (EASE from 0.808 to 0.465) and barely moves ILD. On EASE the recall stays at 0.278 against 0.281, so the NDCG drop is mostly hits sitting lower in the list.

Item-side fairness is the reranker that moves the popularity bands. On EASE the head share falls from 0.876 to 0.636 and the tail share from 0.005 to 0.107, with coverage from 0.237 to 0.365. Men are ahead on validation for every model, so user-side fairness rewrites their 670 lists and leaves the women on the base list. The EASE gender gap falls from 0.044 to 0.015, and the women's NDCG stays at 0.299. The mean NDCG is lower after this rerank. The method spends accuracy the higher group already had.


## Task 3.2 — Evaluation after reranking

We evaluate EASE, SLIMElastic, NGCF, NeuMF and BPR with all four
rerankers, using the validation-selected λ values from Task 3.1.

Each result is compared with the same model's base. Changes are
reported as reranked minus base: increases in ILD, novelty and
coverage indicate increases, while reductions in genre KL,
exposure KL and the gender NDCG gap indicate target improvement.

We report NDCG and recall alongside these measures, exposure
shares and gender-specific NDCG. The selected settings are checked
against the validation NDCG budget of 0.02; this budget is not
guaranteed on test.

In [ ]:
# =============================================================================
# 3.2.1  Accuracy after reranking, relative to each model's base
# =============================================================================
_test = pd.read_csv(RUNS / "tables" / "table_3_1_test.csv")
_valid = pd.read_csv(RUNS / "tables" / "table_3_1_validation.csv")

_models = ["EASE", "SLIMElastic", "NGCF", "NeuMF", "BPR"]
_methods = ["base", "mmr", "calibration", "item_fair", "user_fair"]

_base = (
    _test[_test["method"] == "base"]
    [["label", "ndcg", "recall"]]
    .rename(columns={"ndcg": "base_ndcg", "recall": "base_recall"})
)

_accuracy = _test.merge(
    _base, on="label", validate="many_to_one"
)
_accuracy["delta_ndcg"] = _accuracy["ndcg"] - _accuracy["base_ndcg"]
_accuracy["delta_recall"] = _accuracy["recall"] - _accuracy["base_recall"]

_accuracy["label"] = pd.Categorical(
    _accuracy["label"], categories=_models, ordered=True
)
_accuracy["method"] = pd.Categorical(
    _accuracy["method"], categories=_methods, ordered=True
)

ACCURACY_3_2 = (
    _accuracy.sort_values(["label", "method"])
    [["label", "method", "lam", "ndcg", "delta_ndcg",
      "recall", "delta_recall"]]
    .reset_index(drop=True)
)

with pd.option_context("display.float_format", "{:.4f}".format):
    print(ACCURACY_3_2.to_string(index=False))

# Check the validation budget at the selected λ.
_selected = _test[_test["method"] != "base"][
    ["label", "method", "lam"]
]
_check = _selected.merge(
    _valid[["label", "method", "lam", "ndcg", "base_ndcg"]],
    on=["label", "method", "lam"],
    how="left",
    validate="one_to_one",
)

assert len(_test) == 25, "Expected 25 rows: five models and five methods."
assert _check["ndcg"].notna().all(), "Selected λ missing from validation."

_check["valid_drop"] = _check["base_ndcg"] - _check["ndcg"]
_exceeded = _check[_check["valid_drop"] > 0.02 + 1e-12]

if _exceeded.empty:
    print("\nAll selected settings satisfy the validation NDCG budget.")
else:
    print("\nSelected settings exceeding the validation NDCG budget:")
    print(
        _exceeded[["label", "method", "lam", "valid_drop"]]
        .round(4).to_string(index=False)
    )

ACCURACY_3_2.to_csv(
    RUNS / "tables" / "table_3_2_accuracy.csv", index=False
)

In [ ]:
# =============================================================================
# 3.2.2  Beyond-accuracy changes and their distribution
# =============================================================================
_metrics = ["ild", "novelty", "coverage", "cal_kl", "item_kl", "gap"]

_base = (
    _test[_test["method"] == "base"][["label", *_metrics]]
    .rename(columns={m: f"base_{m}" for m in _metrics})
)

_beyond = _test.merge(
    _base, on="label", validate="many_to_one"
)

for _metric in _metrics:
    _beyond[f"delta_{_metric}"] = (
        _beyond[_metric] - _beyond[f"base_{_metric}"]
    )

_beyond["label"] = pd.Categorical(
    _beyond["label"], categories=_models, ordered=True
)
_beyond["method"] = pd.Categorical(
    _beyond["method"], categories=_methods, ordered=True
)
_beyond = _beyond.sort_values(["label", "method"]).reset_index(drop=True)

BEYOND_3_2 = _beyond[
    ["label", "method",
     "ild", "delta_ild",
     "novelty", "delta_novelty",
     "coverage", "delta_coverage",
     "cal_kl", "delta_cal_kl"]
]

FAIRNESS_3_2 = _beyond[
    ["label", "method",
     "item_kl", "delta_item_kl",
     "head", "mid", "tail",
     "ndcg_F", "ndcg_M", "gap", "delta_gap"]
]

with pd.option_context("display.float_format", "{:.4f}".format):
    print("Diversity, novelty, coverage and calibration")
    print(BEYOND_3_2.to_string(index=False))

    print("\nExposure and gender performance")
    print(FAIRNESS_3_2.to_string(index=False))

BEYOND_3_2.to_csv(
    RUNS / "tables" / "table_3_2_beyond.csv", index=False
)
FAIRNESS_3_2.to_csv(
    RUNS / "tables" / "table_3_2_fairness.csv", index=False
)

### Test trade-offs

Each point uses a validation-selected λ. The horizontal axis
shows test NDCG loss and the vertical axis shows target improvement.
Upper-left points offer more improvement at lower accuracy cost.
Vertical scales differ between panels.

In [ ]:
# =============================================================================
# 3.2.3  Target improvement against NDCG loss
# =============================================================================
_targets = [
    ("mmr", "ild", "Diversification", "ILD increase", 1),
    ("calibration", "cal_kl", "Calibration", "Genre KL reduction", -1),
    ("item_fair", "item_kl", "Item-side fairness", "Exposure KL reduction", -1),
    ("user_fair", "gap", "User-side fairness", "Gender NDCG gap reduction", -1),
]

_base = _test[_test["method"] == "base"].set_index("label")
_colors = dict(zip(_models, plt.get_cmap("tab10").colors))

fig, axes = plt.subplots(2, 2, figsize=(11, 7.5))

for ax, (method, metric, title, ylabel, direction) in zip(
    axes.ravel(), _targets
):
    _rows = (
        _test[_test["method"] == method]
        .set_index("label")
        .reindex(_models)
    )

    for model in _models:
        _loss = _base.loc[model, "ndcg"] - _rows.loc[model, "ndcg"]
        _gain = direction * (
            _rows.loc[model, metric] - _base.loc[model, metric]
        )

        ax.scatter(
            _loss, _gain, color=_colors[model],
            s=65, label=model, zorder=3,
        )

    ax.axhline(0, color="grey", lw=0.8, ls="--")
    ax.axvline(0, color="grey", lw=0.8, ls="--")
    ax.set_title(title)
    ax.set_xlabel("NDCG@10 loss")
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3)

_handles, _labels = axes[0, 0].get_legend_handles_labels()
fig.legend(
    _handles, _labels, loc="upper center",
    ncol=len(_models), frameon=False,
)
fig.tight_layout(rect=[0, 0, 1, 0.93])
fig.savefig(
    RUNS / "tables" / "fig_task32_test.png",
    dpi=150, bbox_inches="tight",
)
plt.show()

### Validation trade-off curves

Points are connected in increasing λ order. Outlined markers
identify selected settings; the vertical line marks the validation
NDCG loss budget of 0.02.

In [ ]:
# =============================================================================
# 3.2.4  Validation trade-offs across the λ grid
# =============================================================================
fig, axes = plt.subplots(2, 2, figsize=(11, 7.5))

for ax, (method, metric, title, ylabel, direction) in zip(
    axes.ravel(), _targets
):
    for model in _models:
        _curve = _valid[
            (_valid["label"] == model)
            & (_valid["method"] == method)
        ].sort_values("lam")

        _base = _curve[_curve["lam"] == 0].iloc[0]
        _loss = _base["ndcg"] - _curve["ndcg"]
        _gain = direction * (_curve[metric] - _base[metric])

        ax.plot(
            _loss, _gain, color=_colors[model],
            marker=".", lw=1.3, label=model,
        )

        _lam = _test.loc[
            (_test["label"] == model)
            & (_test["method"] == method), "lam"
        ].iloc[0]

        _chosen = _curve[
            (_curve["lam"] - _lam).abs() < 1e-12
        ].iloc[0]

        ax.scatter(
            _base["ndcg"] - _chosen["ndcg"],
            direction * (_chosen[metric] - _base[metric]),
            color=_colors[model], edgecolor="black",
            s=75, linewidth=0.8, zorder=4,
        )

    ax.axvline(0.02, color="grey", lw=1, ls="--")
    ax.axhline(0, color="grey", lw=0.8)
    ax.set_title(title)
    ax.set_xlabel("Validation NDCG@10 loss")
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3)

_handles, _labels = axes[0, 0].get_legend_handles_labels()
fig.legend(
    _handles, _labels, loc="upper center",
    ncol=len(_models), frameon=False,
)
fig.tight_layout(rect=[0, 0, 1, 0.93])
fig.savefig(
    RUNS / "tables" / "fig_task32_validation.png",
    dpi=150, bbox_inches="tight",
)
plt.show()

### Reading Task 3.2

All four rerankers improve their intended test objective across
the five models, with a loss in NDCG. The trade-off depends on
the base model and the validation-selected λ.

MMR increases ILD by 0.064–0.094, but coverage sometimes falls:
genre diversity does not necessarily expose more distinct items.
Calibration reduces genre KL across all models. For SLIMElastic
and NGCF, recall increases while NDCG falls, so retrieving more
relevant items does not guarantee better ranking quality.

Item-side fairness increases tail exposure and coverage, while
ILD slightly decreases. User-side fairness narrows the gender
gap by lowering men's NDCG; women's recommendations remain
unchanged.

The validation curves show that stronger intervention generally
buys further target improvement at greater accuracy cost.
For user-side fairness, larger λ values can reverse the gender
ordering and widen the absolute gap. Stronger intervention is therefore not always
better.

All selected settings meet the validation budget, but some test
NDCG losses exceed 0.02. Validation constraints do not guarantee
the same test behaviour.

## Task 3.3 — Reranking before and after hybridisation

We compare two placements of reranking for the weighted,
switching, cascade and mixed hybrids:

1. Rerank → Hybrid: rerank the component recommendations before
   combining them.
2. Hybrid → Rerank: combine the component recommendations first,
   then rerank the hybrid output.

Each pipeline is compared with its corresponding hybrid without
reranking. We consider MMR, calibration, item-side fairness and
user-side fairness, retaining the same final list length of 10.

The comparison examines whether improvements introduced at the
component level survive combination, and whether reranking the
hybrid directly gives better control over the final list.
Accuracy and beyond-accuracy are evaluated using the definitions
from Task 3.2. Reranking parameters are selected on validation
and kept fixed for test evaluation.

### Hybrid configurations

We retain all four hybrid methods rather than carrying forward
only the highest-scoring hybrid. Their configurations are taken
from the earlier validation-based selection.

| Hybrid | Configuration |
|---|---|
| Weighted | Logistic(C=1) weights over EASE, NGCF, ItemKNN-item, BPR and NeuMF |
| Switching | EASE for training profiles of at most 137 interactions; NGCF otherwise |
| Cascade | EASE selects 10 candidates; NGCF orders them |
| Mixed | Alternate between the EASE and NGCF rankings, removing duplicates |


### Passing reranking results into the hybrids

Before hybridisation, each component's top-100 candidates are
reranked. Its existing fusion scores at those positions are
reassigned to the items in the new order. Scores outside the
pool remain unchanged. These scores transfer the reranked order;
they are not new relevance estimates.

The weighted hybrid retains its original Logistic(C=1) coefficients
to control for weight changes. They are not assumed to remain
optimal after score reassignment.

After hybridisation, weighted, switching and mixed provide
100 candidates; cascade retains its 10-item shortlist. Mixed
uses decreasing rank-based relevance scores. Both placements
normalise relevance within the reranking pool, unlike Task 3.1's
normalisation over all legal candidates.

For user fairness, the advantaged gender is determined from each
base hybrid's validation NDCG and fixed across placements and splits.

At λ = 0, the original scores or lists are returned directly.
Checks verify agreement with the base top-10 items and order.
Pre-hybrid reranking uses one shared λ across its components.

Popularity-group boundary assignments are aligned with the
verified earlier results before the hybrid experiments.

In [ ]:
# Match the verified popularity-group boundaries from earlier results.
HEAD = (HEAD - {"66"}) | {"367"}
MID = (MID - {"367", "740"}) | {"66", "60"}
TAIL = set(POP_PROB) - HEAD - MID

# Refresh the features used by item-side fairness.
GROUP_INDEX = np.full(len(ITEMS), 2, dtype=int)

for j, item in enumerate(ITEMS):
    if str(item) in HEAD:
        GROUP_INDEX[j] = 0
    elif str(item) in MID:
        GROUP_INDEX[j] = 1

GROUP_DIST = np.eye(3, dtype=np.float64)[GROUP_INDEX]

assert len(HEAD) == 166
assert len(MID) == 330
assert len(TAIL) == 1159
assert "367" in HEAD and "66" in MID
assert {"60", "387"}.issubset(MID)
assert "740" in TAIL

In [ ]:
# =============================================================================
# 3.3.0  Component scores and fixed hybrid settings
# =============================================================================
from io import StringIO

COMPONENTS_3_3 = [
    "EASE", "NGCF", "ItemKNN-item", "BPR", "NeuMF", "SLIMElastic"
]
WEIGHTED_MODELS_3_3 = [
    "EASE", "NGCF", "ItemKNN-item", "BPR", "NeuMF"
]

SCORES_3_3 = {
    model: load_scores(f"tuned-{model}")
    for model in COMPONENTS_3_3
}

_reference = SCORES_3_3["EASE"]

for model, pack in SCORES_3_3.items():
    for key in ["items", "users_valid", "users_test"]:
        assert np.array_equal(pack[key], _reference[key]), (
            f"{model}: {key} do not match"
        )

    for split in ["valid", "test"]:
        assert np.array_equal(
            np.isfinite(pack[split]),
            np.isfinite(_reference[split]),
        ), f"{model}: {split} candidate masks do not match"

# Read the coefficient section before the relative-weight summary.
_text = (RUNS / "hybrids" / "weights.csv").read_text()
_coefficients = pd.read_csv(
    StringIO(_text.split("# relative magnitudes")[0]),
    comment="#",
    index_col=0,
)

_weights = _coefficients.loc["logistic(C=1.0)"]
WEIGHTS_3_3 = pd.Series(
    {
        model: float(_weights[f"tuned-{model}"])
        for model in WEIGHTED_MODELS_3_3
    },
    name="weight",
)

SWITCH_THRESHOLD_3_3 = 137
CASCADE_N_3_3 = 10

assert np.isfinite(WEIGHTS_3_3.to_numpy()).all()

print("Score matrices and candidate masks agree across components.")
print(f"Loaded {len(SCORES_3_3)} component models.")
print("\nFixed weighted-hybrid coefficients:")
print(WEIGHTS_3_3.round(4).to_string())
print(f"\nSwitching threshold: {SWITCH_THRESHOLD_3_3}")
print(f"Cascade shortlist: {CASCADE_N_3_3}")

In [ ]:
# =============================================================================
# 3.3.1  Reconstruct and verify the four base hybrids
# Reuse Task 1's combination rules; verify base metrics against Task 2.
# Mixed provides 100 candidates while preserving its original top-10.
# =============================================================================
def normalise_3_3(scores):
    # Match the float32 normalisation used in Task 1.
    masked = np.where(np.isfinite(scores), scores, np.nan).astype(np.float32)
    low = np.nanmin(masked, axis=1, keepdims=True)
    high = np.nanmax(masked, axis=1, keepdims=True)
    span = np.where(high > low, high - low, np.float32(1.0))
    return (masked - low) / span


NORM_3_3 = {
    split: {
        model: normalise_3_3(pack[split])
        for model, pack in SCORES_3_3.items()
    }
    for split in ["valid", "test"]
}

_train_profiles = pd.read_csv(
    RUNS / "splits" / "ml-100k.train.tsv", sep="\t", dtype=str
)
PROFILE_SIZE_3_3 = (
    _train_profiles.groupby(_train_profiles.columns[0]).size().to_dict()
)


def hybrid_scores_3_3(norm, users):
    mask = np.isfinite(norm["EASE"])
    scores = {
        model: np.where(mask, values, -np.inf)
        for model, values in norm.items()
    }

    # Weighted: original normalised scores and learned coefficients.
    weighted = np.zeros(mask.shape, dtype=np.float64)
    for model in WEIGHTED_MODELS_3_3:
        weighted += WEIGHTS_3_3[model] * np.nan_to_num(
            norm[model], nan=0.0
        )
    weighted = np.where(mask, weighted, -np.inf)

    # Switching: choose one model from the training-profile length.
    use_ngcf = np.array([
        PROFILE_SIZE_3_3[str(u)] > SWITCH_THRESHOLD_3_3
        for u in users
    ])
    switching = np.where(
        use_ngcf[:, None], scores["NGCF"], scores["EASE"]
    )

    # Cascade: EASE supplies candidates; NGCF scores only those items.
    shortlist = np.argsort(
        -scores["EASE"], axis=1, kind="stable"
    )[:, :CASCADE_N_3_3]

    cascade = np.full(mask.shape, -np.inf)
    rows = np.arange(len(users))[:, None]
    cascade[rows, shortlist] = scores["NGCF"][rows, shortlist]

    # Mixed: interleave rankings, keeping the first 100 distinct items.
    ranks = {
        model: np.argsort(-scores[model], axis=1, kind="stable")
        for model in ["EASE", "NGCF"]
    }
    mixed = np.full(mask.shape, -np.inf)

    for u in range(len(users)):
        picked, seen = [], set()
        pointers = {"EASE": 0, "NGCF": 0}

        while len(picked) < 100:
            progressed = False
            for model in ["EASE", "NGCF"]:
                while pointers[model] < mask.shape[1]:
                    item = ranks[model][u, pointers[model]]
                    pointers[model] += 1
                    if mask[u, item] and item not in seen:
                        picked.append(item)
                        seen.add(item)
                        progressed = True
                        break
                if len(picked) == 100:
                    break
            if not progressed:
                break

        mixed[u, picked] = np.arange(len(picked), 0, -1)

    return {
        "Weighted": weighted,
        "Switching": switching,
        "Cascade": cascade,
        "Mixed": mixed,
    }


BASE_SCORES_3_3 = {
    split: hybrid_scores_3_3(
        NORM_3_3[split], _reference[f"users_{split}"]
    )
    for split in ["valid", "test"]
}

BASE_LISTS_3_3 = {
    split: {
        name: np.argsort(-scores, axis=1, kind="stable")[:, :K]
        for name, scores in hybrids.items()
    }
    for split, hybrids in BASE_SCORES_3_3.items()
}

# Verify the test metrics against the previously reported hybrids.
_published = pd.read_csv(
    RUNS / "tables" / "table_2_2_selected.csv"
).set_index("label")

_labels = {
    "Weighted": "Weighted logistic",
    "Switching": "Switch EASE/NGCF",
    "Cascade": "Cascade EASE→NGCF",
    "Mixed": "Mixed EASE+NGCF",
}

_rows = []
for name, lists in BASE_LISTS_3_3["test"].items():
    metrics = evaluate(lists, USERS_TEST, GT_TEST, PROFILE_TEST)
    reference = _published.loc[_labels[name]]

    for metric in ["ndcg", "recall"]:
        assert abs(metrics[metric] - reference[f"{metric}@10"]) < 1e-6, (
            f"{name}: {metric} differs from Task 2.2"
        )

    _rows.append({
        "hybrid": name,
        "ndcg": metrics["ndcg"],
        "recall": metrics["recall"],
    })

BASE_TABLE_3_3 = pd.DataFrame(_rows)
print(BASE_TABLE_3_3.round(4).to_string(index=False))
print("\nAll four base hybrids match Task 2.2 on NDCG and recall.")

In [ ]:
# =============================================================================
# 3.3.2  Adapt the existing rerankers to variable output lengths
# =============================================================================
def rerank_3_3(
    pool, relevance, method, lam, length,
    profiles=None, advantaged=None,
):
    n_users, n_candidates = pool.shape
    assert relevance.shape == pool.shape
    assert 0 < length <= n_candidates
    assert method in ["mmr", "calibration", "item_fair", "user_fair"]

    if lam == 0:
        return pool[:, :length].copy()

    if method == "calibration":
        assert profiles is not None and len(profiles) == n_users
    if method == "user_fair":
        assert advantaged is not None and len(advantaged) == n_users

    out = np.empty((n_users, length), dtype=int)

    for u in range(n_users):
        cands = pool[u]
        rel = relevance[u]

        # Keep the original list for the group not receiving MMR.
        if method == "user_fair" and not advantaged[u]:
            out[u] = cands[:length]
            continue

        alive = np.ones(n_candidates, dtype=bool)

        if method in ["mmr", "user_fair"]:
            local_sim = SIM[np.ix_(cands, cands)]
            max_sim = np.zeros(n_candidates)
        else:
            if method == "calibration":
                features = GENRE_DIST[cands]
                target = profiles[u]
                alpha = ALPHA_GENRE
            else:
                features = GROUP_DIST[cands]
                target = TARGET_ITEM
                alpha = ALPHA_ITEM

            counts = np.zeros(features.shape[1])

        for t in range(length):
            if method in ["mmr", "user_fair"]:
                penalty = max_sim
            else:
                proposed = counts + features
                q = (proposed + alpha) / (
                    proposed.sum(axis=1, keepdims=True)
                    + alpha * features.shape[1]
                )
                penalty = np.sum(
                    target * np.log2(target / q), axis=1
                )

            score = (1.0 - lam) * rel - lam * penalty
            score[~alive] = -np.inf
            j = int(np.argmax(score))

            out[u, t] = cands[j]
            alive[j] = False

            if method in ["mmr", "user_fair"]:
                max_sim = np.maximum(max_sim, local_sim[j])
            else:
                counts += features[j]

    return out

In [ ]:
# =============================================================================
# 3.3.3  Prepare candidate pools and check the adapted rerankers
# Check candidate validity, λ = 0, and sample agreement with Task 3.1.
# =============================================================================
def prepare_pool_3_3(scores, pool_size=100):
    finite = np.isfinite(scores)
    assert np.all(finite.sum(axis=1) >= pool_size), (
        "Not enough legal candidates for the requested pool."
    )

    pool = np.argsort(
        -np.where(finite, scores, -np.inf),
        axis=1, kind="stable",
    )[:, :pool_size]

    values = np.take_along_axis(scores, pool, axis=1)
    low = values.min(axis=1, keepdims=True)
    high = values.max(axis=1, keepdims=True)
    span = np.where(high > low, high - low, 1.0)
    relevance = (values - low) / span

    return pool, relevance


# Check candidate preparation, including the cascade's smaller pool.
for name, scores in BASE_SCORES_3_3["valid"].items():
    size = CASCADE_N_3_3 if name == "Cascade" else 100
    pool, relevance = prepare_pool_3_3(scores, size)

    assert np.isfinite(relevance).all()
    assert np.all((relevance >= 0) & (relevance <= 1))
    assert np.array_equal(
        pool[:, :K], BASE_LISTS_3_3["valid"][name]
    )

    for method in ["mmr", "calibration", "item_fair", "user_fair"]:
        unchanged = rerank_3_3(
            pool, relevance, method, lam=0, length=K
        )
        assert np.array_equal(unchanged, pool[:, :K])

print("Candidate pools are valid; λ = 0 preserves all base lists.")

# Compare with Task 3.1 using exactly the same inputs.
_sample = 12
_lam = 0.2
_raw = SCORES_3_3["EASE"]["valid"][:_sample]

_full_rel, _pool = candidate_pool(_raw)
_rel = np.take_along_axis(_full_rel, _pool, axis=1)
_profiles = PROFILE_VALID[:_sample]

# An artificial mask checks both branches of user-side fairness.
_advantaged = np.arange(_sample) % 2 == 0
_mmr = mmr_lists(_pool, _full_rel, _lam)

_expected = {
    "mmr": _mmr,
    "calibration": kl_lists(
        _pool, _full_rel, GENRE_DIST,
        _profiles, _lam, ALPHA_GENRE,
    ),
    "item_fair": kl_lists(
        _pool, _full_rel, GROUP_DIST,
        TARGET_ITEM, _lam, ALPHA_ITEM,
    ),
    "user_fair": user_fair_lists(
        _pool[:, :K], _mmr, _advantaged
    ),
}

for method, expected in _expected.items():
    actual = rerank_3_3(
        _pool, _rel, method, _lam, length=K,
        profiles=_profiles, advantaged=_advantaged,
    )
    assert np.array_equal(actual, expected), (
        f"{method}: adapted output differs from Task 3.1"
    )

print("All four adapted rerankers match Task 3.1 on the sample.")

In [ ]:
# =============================================================================
# 3.3.4  Transfer reranked positions into the component's fusion scores
# =============================================================================
def reassign_scores_3_3(
    scores, method, lam, profiles=None, advantaged=None,
):
    if lam == 0:
        return scores.copy()

    pool, relevance = prepare_pool_3_3(scores, pool_size=100)

    reordered = rerank_3_3(
        pool, relevance, method, lam, length=pool.shape[1],
        profiles=profiles, advantaged=advantaged,
    )

    # Keep the original position scores; only change their item assignment.
    position_scores = np.take_along_axis(scores, pool, axis=1)
    adjusted = scores.copy()
    rows = np.arange(scores.shape[0])[:, None]
    adjusted[rows, reordered] = position_scores

    return adjusted


# λ = 0 must preserve every component score on both splits.
for split, components in NORM_3_3.items():
    for model, scores in components.items():
        for method in ["mmr", "calibration", "item_fair", "user_fair"]:
            adjusted = reassign_scores_3_3(scores, method, lam=0)
            assert np.array_equal(adjusted, scores, equal_nan=True), (
                f"{split}, {model}, {method}: λ = 0 changed scores"
            )

print("λ = 0 preserves all component scores on both splits.")

# Check non-zero reassignment on a small validation sample.
_sample = 12
_scores = NORM_3_3["valid"]["EASE"][:_sample]
_pool, _rel = prepare_pool_3_3(_scores)
_rows = np.arange(_sample)[:, None]

_outside = np.ones(_scores.shape, dtype=bool)
_outside[_rows, _pool] = False

for method in ["mmr", "calibration", "item_fair", "user_fair"]:
    adjusted = reassign_scores_3_3(
        _scores, method, lam=0.2,
        profiles=PROFILE_VALID[:_sample],
        advantaged=np.arange(_sample) % 2 == 0,
    )

    assert np.array_equal(
        np.isfinite(adjusted), np.isfinite(_scores)
    ), f"{method}: candidate mask changed"

    assert np.array_equal(
        adjusted[_outside], _scores[_outside], equal_nan=True
    ), f"{method}: scores outside the pool changed"

    assert np.array_equal(
        np.sort(np.take_along_axis(adjusted, _pool, axis=1), axis=1),
        np.sort(np.take_along_axis(_scores, _pool, axis=1), axis=1),
    ), f"{method}: the pool's score distribution changed"

print("Reassignment preserves masks, outside scores and pool score values.")

In [ ]:
# =============================================================================
# 3.3.5  Connect the two reranking placements
# =============================================================================
HYBRID_COMPONENTS_3_3 = {
    "Weighted": WEIGHTED_MODELS_3_3,
    "Switching": ["EASE", "NGCF"],
    "Cascade": ["EASE", "NGCF"],
    "Mixed": ["EASE", "NGCF"],
}

# Fix the advantaged group using each base hybrid's validation results.
ADVANTAGED_3_3 = {}

for name, lists in BASE_LISTS_3_3["valid"].items():
    metrics = evaluate(
        lists, USERS_VALID, GT_VALID, PROFILE_VALID
    )
    ADVANTAGED_3_3[name] = (
        "M" if metrics["ndcg_M"] >= metrics["ndcg_F"] else "F"
    )

    print(
        f"{name}: validation NDCG "
        f"F={metrics['ndcg_F']:.4f}, M={metrics['ndcg_M']:.4f}; "
        f"group ahead={ADVANTAGED_3_3[name]}"
    )


def pipeline_3_3(split, name, placement, method, lam):
    assert split in ["valid", "test"]
    assert name in HYBRID_COMPONENTS_3_3
    assert placement in ["pre", "post"]

    if lam == 0:
        return BASE_LISTS_3_3[split][name].copy()

    users = _reference[f"users_{split}"]
    profiles = PROFILE_VALID if split == "valid" else PROFILE_TEST
    advantaged = np.array([
        GENDER[str(u)] == ADVANTAGED_3_3[name] for u in users
    ])

    if placement == "pre":
        adjusted = NORM_3_3[split].copy()

        for model in HYBRID_COMPONENTS_3_3[name]:
            adjusted[model] = reassign_scores_3_3(
                NORM_3_3[split][model], method, lam,
                profiles=profiles, advantaged=advantaged,
            )

        scores = hybrid_scores_3_3(adjusted, users)[name]
        return np.argsort(-scores, axis=1, kind="stable")[:, :K]

    scores = BASE_SCORES_3_3[split][name]
    size = CASCADE_N_3_3 if name == "Cascade" else 100
    pool, relevance = prepare_pool_3_3(scores, size)

    return rerank_3_3(
        pool, relevance, method, lam, length=K,
        profiles=profiles, advantaged=advantaged,
    )


def check_lists_3_3(lists, split):
    legal = np.isfinite(NORM_3_3[split]["EASE"])

    assert lists.shape == (legal.shape[0], K)
    assert np.all(np.diff(np.sort(lists, axis=1), axis=1) > 0), (
        "Duplicate items in a recommendation list."
    )
    assert np.take_along_axis(legal, lists, axis=1).all(), (
        "A recommendation contains a masked item."
    )


# Check the explicit λ = 0 behaviour for every pipeline.
for split in ["valid", "test"]:
    for name in HYBRID_COMPONENTS_3_3:
        for placement in ["pre", "post"]:
            for method in ["mmr", "calibration", "item_fair", "user_fair"]:
                lists = pipeline_3_3(
                    split, name, placement, method, lam=0
                )
                check_lists_3_3(lists, split)

                assert np.array_equal(
                    lists, BASE_LISTS_3_3[split][name]
                ), f"{split}, {name}, {placement}, {method}: base mismatch"

print("\nBoth placements recover every base top-10 at λ = 0.")

### Selecting λ on validation

We search the λ grid from Task 3.1, including λ = 0.
Each hybrid, placement and reranker is selected separately.

A setting is eligible if its validation NDCG falls by no more
than 0.02 from the corresponding base hybrid. Among eligible
settings, we select the best value of the intended objective,
breaking ties in favour of the smaller λ. If no setting improves
the objective, the base is retained.

List validity and the cascade and user-fairness restrictions
are checked during the search. Test results are not used.

In [ ]:
# =============================================================================
# 3.3.6  Select λ on validation, checking lists during the search
# =============================================================================
LAMBDA_GRID_3_3 = [0.0, *GRID]

# Direction: +1 means maximise; -1 means minimise.
OBJECTIVES_3_3 = {
    "mmr": ("ild", 1),
    "calibration": ("cal_kl", -1),
    "item_fair": ("item_kl", -1),
    "user_fair": ("gap", -1),
}

_curve_rows = []
_selected_rows = []
_gender = np.array([GENDER[str(u)] for u in USERS_VALID])

for name in HYBRID_COMPONENTS_3_3:
    base = BASE_LISTS_3_3["valid"][name]
    base_metrics = evaluate(
        base, USERS_VALID, GT_VALID, PROFILE_VALID
    )

    for placement in ["pre", "post"]:
        for method, (target, direction) in OBJECTIVES_3_3.items():
            records = []

            for lam in LAMBDA_GRID_3_3:
                if lam == 0:
                    lists = base
                    metrics = base_metrics
                else:
                    lists = pipeline_3_3(
                        "valid", name, placement, method, lam
                    )
                    metrics = evaluate(
                        lists, USERS_VALID, GT_VALID, PROFILE_VALID
                    )

                check_lists_3_3(lists, "valid")

                if name == "Cascade" and placement == "post":
                    assert np.array_equal(
                        np.sort(lists, axis=1),
                        np.sort(base, axis=1),
                    ), "Post-cascade changed its candidate set."

                if method == "user_fair":
                    unchanged = _gender != ADVANTAGED_3_3[name]
                    assert np.array_equal(
                        lists[unchanged], base[unchanged]
                    ), "User fairness changed the other group's lists."

                row = {
                    "hybrid": name,
                    "placement": placement,
                    "method": method,
                    "lam": lam,
                    "base_ndcg": base_metrics["ndcg"],
                    "target": target,
                    "target_improvement": direction * (
                        metrics[target] - base_metrics[target]
                    ),
                    "lists_changed": int(
                        np.any(lists != base, axis=1).sum()
                    ),
                }
                row.update({
                    key: value for key, value in metrics.items()
                    if not key.startswith("_")
                })
                records.append(row)

            # λ = 0 guarantees an eligible fallback.
            eligible = [
                row for row in records
                if row["ndcg"] >= base_metrics["ndcg"] - BUDGET - 1e-12
            ]
            improving = [
                row for row in eligible
                if row["target_improvement"] > 1e-12
            ]

            if improving:
                chosen = min(
                    improving,
                    key=lambda row: (
                        -direction * row[target], row["lam"]
                    ),
                )
            else:
                chosen = records[0]

            _curve_rows.extend(records)
            _selected_rows.append(chosen.copy())

            print(
                f"{name:<10} {placement:<4} {method:<12} "
                f"selected λ={chosen['lam']:.2f}",
                flush=True,
            )

VALIDATION_3_3 = pd.DataFrame(_curve_rows)
SELECTED_3_3 = pd.DataFrame(_selected_rows)

assert len(SELECTED_3_3) == 32
assert (
    SELECTED_3_3["ndcg"]
    >= SELECTED_3_3["base_ndcg"] - BUDGET - 1e-12
).all()

_columns = [
    "hybrid", "placement", "method", "lam",
    "ndcg", "recall", "target", "target_improvement",
]

with pd.option_context("display.float_format", "{:.4f}".format):
    print("\nValidation-selected settings")
    print(SELECTED_3_3[_columns].to_string(index=False))

VALIDATION_3_3.to_csv(
    RUNS / "tables" / "table_3_3_validation.csv", index=False
)
SELECTED_3_3.to_csv(
    RUNS / "tables" / "table_3_3_selected.csv", index=False
)

### Matched-λ validation comparison

We compare placements at the same hybrid, reranker and λ.
Differences are post minus pre; positive values favour post
on the corresponding measure.

The figure shows NDCG differences above and target-improvement
differences below. Outlined markers indicate settings where both
placements satisfy the validation budget.

In [ ]:
# =============================================================================
# 3.3.7  Compare pre and post at the same λ
# =============================================================================
_keys = ["hybrid", "method", "lam"]
_values = ["ndcg", "recall", "target_improvement"]

_pre = VALIDATION_3_3[
    VALIDATION_3_3["placement"] == "pre"
][_keys + _values]

_post = VALIDATION_3_3[
    VALIDATION_3_3["placement"] == "post"
][_keys + _values]

MATCHED_3_3 = _pre.merge(
    _post,
    on=_keys,
    suffixes=("_pre", "_post"),
    validate="one_to_one",
)

for metric in _values:
    MATCHED_3_3[f"delta_{metric}"] = (
        MATCHED_3_3[f"{metric}_post"]
        - MATCHED_3_3[f"{metric}_pre"]
    )

# Attach the base NDCG to identify settings feasible for both placements.
_base_ndcg = (
    VALIDATION_3_3[["hybrid", "base_ndcg"]]
    .drop_duplicates()
)
MATCHED_3_3 = MATCHED_3_3.merge(
    _base_ndcg, on="hybrid", validate="many_to_one"
)

for placement in ["pre", "post"]:
    MATCHED_3_3[f"within_budget_{placement}"] = (
        MATCHED_3_3[f"ndcg_{placement}"]
        >= MATCHED_3_3["base_ndcg"] - BUDGET - 1e-12
    )

MATCHED_3_3 = MATCHED_3_3.sort_values(
    ["hybrid", "method", "lam"]
).reset_index(drop=True)

assert len(MATCHED_3_3) == 160

_columns = [
    "hybrid", "method", "lam",
    "delta_ndcg", "delta_recall", "delta_target_improvement",
    "within_budget_pre", "within_budget_post",
]

with pd.option_context("display.float_format", "{:.4f}".format):
    print(MATCHED_3_3[_columns].to_string(index=False))

MATCHED_3_3.to_csv(
    RUNS / "tables" / "table_3_3_matched.csv", index=False
)

In [ ]:
# =============================================================================
# 3.3.8  Placement differences at matched λ values
# =============================================================================
_matched = pd.read_csv(
    RUNS / "tables" / "table_3_3_matched.csv"
)

_methods = [
    ("mmr", "Diversification", "Difference in ILD improvement"),
    ("calibration", "Calibration", "Difference in genre KL reduction"),
    ("item_fair", "Item-side fairness", "Difference in exposure KL reduction"),
    ("user_fair", "User-side fairness", "Difference in gender gap reduction"),
]
_hybrids = ["Weighted", "Switching", "Cascade", "Mixed"]
_colors = dict(zip(_hybrids, plt.get_cmap("tab10").colors))

fig, axes = plt.subplots(2, 4, figsize=(16, 7))

for col, (method, title, ylabel) in enumerate(_methods):
    for name in _hybrids:
        rows = _matched[
            (_matched["hybrid"] == name)
            & (_matched["method"] == method)
        ].sort_values("lam")

        feasible = (
            rows["within_budget_pre"] & rows["within_budget_post"]
        )

        for row, metric in enumerate([
            "delta_ndcg", "delta_target_improvement"
        ]):
            ax = axes[row, col]
            ax.plot(
                rows["lam"], rows[metric],
                color=_colors[name], marker=".", label=name,
            )
            ax.scatter(
                rows.loc[feasible, "lam"],
                rows.loc[feasible, metric],
                facecolors="none", edgecolors=_colors[name],
                s=65, zorder=3,
            )

    axes[0, col].set_title(title)
    axes[0, col].set_ylabel("NDCG difference (post − pre)")
    axes[1, col].set_ylabel(ylabel)

for ax in axes.ravel():
    ax.axhline(0, color="grey", lw=0.8, ls="--")
    ax.set_xlabel("λ")
    ax.grid(alpha=0.3)

_handles, _labels = axes[0, 0].get_legend_handles_labels()
fig.legend(
    _handles, _labels, loc="upper center",
    ncol=len(_hybrids), frameon=False,
)
fig.tight_layout(rect=[0, 0, 1, 0.93])
fig.savefig(
    RUNS / "tables" / "fig_task33_matched.png",
    dpi=150, bbox_inches="tight",
)
plt.show()

### Test comparison at selected settings

Each placement uses its own validation-selected λ, fixed before
test evaluation. Results are compared with the corresponding base;
differences reflect the tuned pipelines, including their parameters
and candidate restrictions.

In the trade-off figure, colour identifies the hybrid, circles
denote pre and triangles denote post. The axes show NDCG loss
and target improvement. Lines connect placements, not λ values.

In [ ]:
# =============================================================================
# 3.3.9  Test evaluation with validation-selected λ
# =============================================================================
_selected = pd.read_csv(
    RUNS / "tables" / "table_3_3_selected.csv"
)

_test_rows = []
_base_metrics = {}
TEST_LISTS_3_3 = {}

for name in HYBRID_COMPONENTS_3_3:
    lists = BASE_LISTS_3_3["test"][name]
    metrics = evaluate(
        lists, USERS_TEST, GT_TEST, PROFILE_TEST
    )
    _base_metrics[name] = metrics
    TEST_LISTS_3_3[(name, "base", "base")] = lists.copy()

    row = {
        "hybrid": name,
        "placement": "base",
        "method": "base",
        "lam": 0.0,
        "delta_ndcg": 0.0,
        "delta_recall": 0.0,
        "target_improvement": 0.0,
        "lists_changed": 0,
    }
    row.update({
        key: value for key, value in metrics.items()
        if not key.startswith("_")
    })
    _test_rows.append(row)

_gender = np.array([GENDER[str(u)] for u in USERS_TEST])

for setting in _selected.itertuples(index=False):
    name = setting.hybrid
    placement = setting.placement
    method = setting.method

    lists = pipeline_3_3(
        "test", name, placement, method, setting.lam
    )
    check_lists_3_3(lists, "test")
    base = BASE_LISTS_3_3["test"][name]

    if name == "Cascade" and placement == "post":
        assert np.array_equal(
            np.sort(lists, axis=1), np.sort(base, axis=1)
        )

    if method == "user_fair":
        unchanged = _gender != ADVANTAGED_3_3[name]
        assert np.array_equal(lists[unchanged], base[unchanged])

    metrics = evaluate(
        lists, USERS_TEST, GT_TEST, PROFILE_TEST
    )
    target, direction = OBJECTIVES_3_3[method]

    row = {
        "hybrid": name,
        "placement": placement,
        "method": method,
        "lam": setting.lam,
        "delta_ndcg": metrics["ndcg"] - _base_metrics[name]["ndcg"],
        "delta_recall": metrics["recall"] - _base_metrics[name]["recall"],
        "target_improvement": direction * (
            metrics[target] - _base_metrics[name][target]
        ),
        "lists_changed": int(np.any(lists != base, axis=1).sum()),
    }
    row.update({
        key: value for key, value in metrics.items()
        if not key.startswith("_")
    })
    _test_rows.append(row)
    TEST_LISTS_3_3[(name, placement, method)] = lists.copy()

    print(f"Evaluated {name}: {placement}, {method}", flush=True)

TEST_3_3 = pd.DataFrame(_test_rows)
assert len(TEST_3_3) == 36

_columns = [
    "hybrid", "placement", "method", "lam",
    "ndcg", "delta_ndcg", "recall", "delta_recall",
    "target_improvement",
]

with pd.option_context("display.float_format", "{:.4f}".format):
    print(TEST_3_3[_columns].to_string(index=False))

TEST_3_3.to_csv(
    RUNS / "tables" / "table_3_3_test.csv", index=False
)

_saved_lists = {
    f"{name}__{placement}__{method}": lists
    for (name, placement, method), lists in TEST_LISTS_3_3.items()
}

np.savez_compressed(
    RUNS / "rerank" / "hybrid_3_3_test_lists.npz",
    users=USERS_TEST,
    items=ITEMS,
    **_saved_lists,
)

In [ ]:
# =============================================================================
# 3.3.10  Side-by-side test comparison of the selected pipelines
# =============================================================================
_test_results = pd.read_csv(
    RUNS / "tables" / "table_3_3_test.csv"
)

_keys = ["hybrid", "method"]
_values = [
    "lam", "ndcg", "delta_ndcg",
    "recall", "target_improvement",
]

_pre = _test_results[
    _test_results["placement"] == "pre"
][_keys + _values]

_post = _test_results[
    _test_results["placement"] == "post"
][_keys + _values]

COMPARISON_3_3 = _pre.merge(
    _post,
    on=_keys,
    suffixes=("_pre", "_post"),
    validate="one_to_one",
)

_hybrids = ["Weighted", "Switching", "Cascade", "Mixed"]
_methods = ["mmr", "calibration", "item_fair", "user_fair"]

COMPARISON_3_3["hybrid"] = pd.Categorical(
    COMPARISON_3_3["hybrid"], categories=_hybrids, ordered=True
)
COMPARISON_3_3["method"] = pd.Categorical(
    COMPARISON_3_3["method"], categories=_methods, ordered=True
)
COMPARISON_3_3 = COMPARISON_3_3.sort_values(
    _keys
).reset_index(drop=True)

assert len(COMPARISON_3_3) == 16

_accuracy_columns = [
    "hybrid", "method", "lam_pre", "lam_post",
    "ndcg_pre", "ndcg_post",
    "delta_ndcg_pre", "delta_ndcg_post",
]

_target_columns = [
    "hybrid", "method",
    "recall_pre", "recall_post",
    "target_improvement_pre", "target_improvement_post",
]

with pd.option_context("display.float_format", "{:.4f}".format):
    print("Selected λ and test accuracy")
    print(COMPARISON_3_3[_accuracy_columns].to_string(index=False))

    print("\nRecall and target improvement")
    print(COMPARISON_3_3[_target_columns].to_string(index=False))

In [ ]:
# =============================================================================
# 3.3.11  Test trade-offs at the validation-selected settings
# =============================================================================
_test_results = pd.read_csv(
    RUNS / "tables" / "table_3_3_test.csv"
)

_methods = [
    ("mmr", "Diversification", "ILD increase"),
    ("calibration", "Calibration", "Genre KL reduction"),
    ("item_fair", "Item-side fairness", "Exposure KL reduction"),
    ("user_fair", "User-side fairness", "Gender NDCG gap reduction"),
]
_hybrids = ["Weighted", "Switching", "Cascade", "Mixed"]
_colors = dict(zip(_hybrids, plt.get_cmap("tab10").colors))
_markers = {"pre": "o", "post": "^"}

fig, axes = plt.subplots(2, 2, figsize=(11, 7.5))

for ax, (method, title, ylabel) in zip(axes.ravel(), _methods):
    for name in _hybrids:
        rows = (
            _test_results[
                (_test_results["hybrid"] == name)
                & (_test_results["method"] == method)
            ]
            .set_index("placement")
            .loc[["pre", "post"]]
        )

        ax.plot(
            -rows["delta_ndcg"], rows["target_improvement"],
            color=_colors[name], lw=1, alpha=0.5,
        )

        for placement, marker in _markers.items():
            point = rows.loc[placement]
            ax.scatter(
                -point["delta_ndcg"],
                point["target_improvement"],
                color=_colors[name], marker=marker,
                s=75, zorder=3,
            )

    ax.axhline(0, color="grey", lw=0.8, ls="--")
    ax.axvline(0, color="grey", lw=0.8, ls="--")
    ax.set_title(title)
    ax.set_xlabel("Test NDCG@10 loss")
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3)

# Separate legends for hybrid colours and placement markers.
from matplotlib.lines import Line2D

_handles = [
    Line2D([0], [0], color=_colors[name], lw=2, label=name)
    for name in _hybrids
]
_handles += [
    Line2D(
        [0], [0], color="black", marker=marker,
        linestyle="None", markersize=7, label=placement,
    )
    for placement, marker in _markers.items()
]

fig.legend(
    handles=_handles, loc="upper center",
    ncol=6, frameon=False,
)
fig.tight_layout(rect=[0, 0, 1, 0.93])
fig.savefig(
    RUNS / "tables" / "fig_task33_test.png",
    dpi=150, bbox_inches="tight",
)
plt.show()

### Reading Task 3.3

Placement effects depend on the hybrid's combination rule.

For the weighted hybrid, post-hybrid reranking improves all four
targets more, but incurs larger NDCG losses. Controlling the final
list gives stronger target alignment at a higher accuracy cost.

Switching produces identical metrics across placements on the
validation grid and at the selected test settings. Each user
receives one component's ranking, so placement makes no observed
difference here.

Cascade's post-hybrid pool contains only 10 items. Reordering
cannot improve ILD or either KL objective, so these methods
retain the base. Pre-hybrid reranking can change shortlist membership.

Mixed has no consistent winner. Pre-hybrid MMR improves ILD more
with less NDCG loss; other methods show trade-offs.

Test settings use separately selected λ values. Matched-λ validation
comparisons provide additional context, while candidate restrictions
and score reassignment remain relevant limitations.

## Task 3.4 — Reranking effects across user and item groups

We evaluate all individual models and hybrid placements against
their own bases within each group.

User groups are training-profile length (low ≤34, mid 35–92,
high >92) and gender. Item groups use training popularity:
head, mid and tail. Items without training interactions belong
to the tail. All systems use the same group assignments within
this evaluation.

The analysis distinguishes target improvements from accuracy
costs, and increased item exposure from improved retrieval.

In [ ]:
# =============================================================================
# 3.4.0  User and item groups, retaining the earlier definitions
# =============================================================================
USER_GROUPS_3_4 = pd.DataFrame({
    "user": USERS_TEST,
    "profile_size": [
        PROFILE_SIZE_3_3[str(u)] for u in USERS_TEST
    ],
    "gender": [
        GENDER[str(u)] for u in USERS_TEST
    ],
})

USER_GROUPS_3_4["profile_group"] = pd.cut(
    USER_GROUPS_3_4["profile_size"],
    bins=[-1, 34, 92, np.inf],
    labels=["low", "mid", "high"],
)

# Restrict item-group summaries to actual catalogue items.
ITEM_GROUPS_3_4 = pd.DataFrame({
    "item": [str(item) for item in ITEMS if str(item) in CATALOG]
})

ITEM_GROUPS_3_4["popularity_group"] = [
    "head" if item in HEAD else "mid" if item in MID else "tail"
    for item in ITEM_GROUPS_3_4["item"]
]
ITEM_GROUPS_3_4["popularity_group"] = pd.Categorical(
    ITEM_GROUPS_3_4["popularity_group"],
    categories=["head", "mid", "tail"],
    ordered=True,
)

assert USER_GROUPS_3_4["profile_group"].notna().all()
assert USER_GROUPS_3_4["user"].is_unique
assert ITEM_GROUPS_3_4["item"].is_unique

print("Users by profile length")
print(
    USER_GROUPS_3_4.groupby("profile_group", observed=True)
    .agg(users=("user", "size"), mean_profile=("profile_size", "mean"))
    .round(1).to_string()
)

print("\nUsers by gender")
print(USER_GROUPS_3_4["gender"].value_counts().to_string())

print("\nItems by popularity")
print(
    ITEM_GROUPS_3_4.groupby("popularity_group", observed=True)
    .size().to_string()
)

USER_GROUPS_3_4.to_csv(
    RUNS / "tables" / "table_3_4_user_groups.csv", index=False
)
ITEM_GROUPS_3_4.to_csv(
    RUNS / "tables" / "table_3_4_item_groups.csv", index=False
)

In [ ]:
# =============================================================================
# 3.4.1  Load and align the saved test recommendation lists
# Align saved test lists and link each configuration to its own base.
# =============================================================================
LISTS_3_4 = {}
_system_rows = []
_item_column = {str(item): j for j, item in enumerate(ITEMS)}


def read_lists_3_4(path):
    recommendations = json.loads(path.read_text())["recommendations"]
    assert set(recommendations) == set(map(str, USERS_TEST)), (
        f"{path.name}: test users do not match"
    )

    return np.array([
        [
            _item_column[str(item)]
            for item in recommendations[str(user)]["items"][:K]
        ]
        for user in USERS_TEST
    ], dtype=int)


def register_3_4(system, family, model, placement, method, base, lists):
    check_lists_3_3(lists, "test")
    assert system not in LISTS_3_4, "Duplicate system ID."

    LISTS_3_4[system] = lists
    _system_rows.append({
        "system": system,
        "family": family,
        "model": model,
        "placement": placement,
        "method": method,
        "base": base,
    })


# Individual models: one base and four reranked lists per model.
_models = ["EASE", "SLIMElastic", "NGCF", "NeuMF", "BPR"]
_methods = ["mmr", "calibration", "item_fair", "user_fair"]

for model in _models:
    base = f"individual::{model}::base"
    lists = read_lists_3_4(
        RUNS / "lists" / f"tuned-{model}_top10.json"
    )
    register_3_4(
        base, "individual", model, "base", "base", base, lists
    )

    for method in _methods:
        lists = read_lists_3_4(
            RUNS / "rerank" / f"{model}__{method}_top10.json"
        )
        register_3_4(
            f"individual::{model}::{method}",
            "individual", model, "post", method, base, lists,
        )

# Hybrids: one base and eight reranked lists per hybrid.
with np.load(
    RUNS / "rerank" / "hybrid_3_3_test_lists.npz"
) as saved:
    assert np.array_equal(saved["users"], USERS_TEST)
    assert np.array_equal(saved["items"], ITEMS)

    for model in ["Weighted", "Switching", "Cascade", "Mixed"]:
        base = f"hybrid::{model}::base"

        register_3_4(
            base, "hybrid", model, "base", "base", base,
            saved[f"{model}__base__base"],
        )

        for placement in ["pre", "post"]:
            for method in _methods:
                register_3_4(
                    f"hybrid::{model}::{placement}::{method}",
                    "hybrid", model, placement, method, base,
                    saved[f"{model}__{placement}__{method}"],
                )

SYSTEMS_3_4 = pd.DataFrame(_system_rows)
assert len(SYSTEMS_3_4) == 61
assert SYSTEMS_3_4["base"].isin(LISTS_3_4).all()

print(
    SYSTEMS_3_4.groupby("family")
    .size().rename("systems").to_string()
)
print("\nAll 61 systems are aligned and linked to their own bases.")

SYSTEMS_3_4.to_csv(
    RUNS / "tables" / "table_3_4_systems.csv", index=False
)

### Group-level evaluation

User metrics are averaged within each profile-length or gender
group. Item-group recall is averaged only over users with held-out
items in that group; exposure is the share of recommendation slots,
and coverage is the fraction of group items shown.

Changes are relative to the same base and group. Profile groups
also differ in held-out set size, which affects recall comparisons.

In [ ]:
# =============================================================================
# 3.4.2  Per-user metrics and changes within user groups
# =============================================================================
def user_metrics_3_4(lists):
    discount = 1.0 / np.log2(np.arange(1, K + 1) + 1.0)
    ndcg = []
    recall = []

    for user, cols in zip(USERS_TEST, lists):
        relevant = GT_TEST[str(user)]
        hits = np.array([
            float(str(ITEMS[j]) in relevant) for j in cols
        ])
        ideal = discount[:min(len(relevant), K)].sum()
        ndcg.append(float((hits * discount).sum() / ideal))
        recall.append(float(hits.sum() / len(relevant)))

    # Mean genre distance over distinct item pairs.
    similarity = SIM[lists[:, :, None], lists[:, None, :]]
    upper = np.triu_indices(K, k=1)
    ild = (1.0 - similarity[:, upper[0], upper[1]]).mean(axis=1)

    counts = GENRE_DIST[lists].sum(axis=1)
    q = counts + ALPHA_GENRE
    q /= q.sum(axis=1, keepdims=True)
    cal_kl = np.sum(
        PROFILE_TEST * np.log2(PROFILE_TEST / q), axis=1
    )

    return pd.DataFrame({
        "user": USERS_TEST,
        "ndcg": ndcg,
        "recall": recall,
        "ild": ild,
        "cal_kl": cal_kl,
    })


_user_rows = []

for system, lists in LISTS_3_4.items():
    frame = user_metrics_3_4(lists)
    frame["system"] = system
    _user_rows.append(frame)

PER_USER_3_4 = pd.concat(_user_rows, ignore_index=True)
PER_USER_3_4 = PER_USER_3_4.merge(
    USER_GROUPS_3_4, on="user", validate="many_to_one"
)

_metrics = ["ndcg", "recall", "ild", "cal_kl"]
_group_rows = []

for group_type, column in [
    ("profile", "profile_group"),
    ("gender", "gender"),
]:
    grouped = (
        PER_USER_3_4.groupby(["system", column], observed=True)
        .agg(
            users=("user", "size"),
            **{metric: (metric, "mean") for metric in _metrics},
        )
        .reset_index()
        .rename(columns={column: "group"})
    )
    grouped["group"] = grouped["group"].astype(str)
    grouped["group_type"] = group_type
    _group_rows.append(grouped)

USER_RESULTS_3_4 = pd.concat(_group_rows, ignore_index=True)
USER_RESULTS_3_4 = USER_RESULTS_3_4.merge(
    SYSTEMS_3_4, on="system", validate="many_to_one"
)

# Match each group result to the same group under its own base.
_base = USER_RESULTS_3_4[
    USER_RESULTS_3_4["method"] == "base"
][["system", "group_type", "group", *_metrics]].rename(
    columns={
        "system": "base",
        **{metric: f"base_{metric}" for metric in _metrics},
    }
)

USER_RESULTS_3_4 = USER_RESULTS_3_4.merge(
    _base,
    on=["base", "group_type", "group"],
    how="left",
    validate="many_to_one",
)

for metric in _metrics:
    assert USER_RESULTS_3_4[f"base_{metric}"].notna().all()
    USER_RESULTS_3_4[f"delta_{metric}"] = (
        USER_RESULTS_3_4[metric]
        - USER_RESULTS_3_4[f"base_{metric}"]
    )

assert len(PER_USER_3_4) == len(LISTS_3_4) * len(USERS_TEST)
assert len(USER_RESULTS_3_4) == 305

print(f"Per-user results: {len(PER_USER_3_4)} rows")
print(f"User-group results: {len(USER_RESULTS_3_4)} rows")

# Show user fairness by gender; all other results remain in the table.
_columns = [
    "family", "model", "placement", "group", "users",
    "ndcg", "delta_ndcg", "recall", "delta_recall",
]
_fairness = USER_RESULTS_3_4[
    (USER_RESULTS_3_4["group_type"] == "gender")
    & (USER_RESULTS_3_4["method"] == "user_fair")
]

with pd.option_context("display.float_format", "{:.4f}".format):
    print("\nUser fairness: performance by gender")
    print(_fairness[_columns].to_string(index=False))

PER_USER_3_4.to_csv(
    RUNS / "tables" / "table_3_4_per_user.csv", index=False
)
USER_RESULTS_3_4.to_csv(
    RUNS / "tables" / "table_3_4_users.csv", index=False
)

In [ ]:
# =============================================================================
# 3.4.3  Exposure, coverage and recall by item-popularity group
# =============================================================================
_groups = ["head", "mid", "tail"]
_item_column = {str(item): j for j, item in enumerate(ITEMS)}

ITEM_SETS_3_4 = {
    group: set(
        ITEM_GROUPS_3_4.loc[
            ITEM_GROUPS_3_4["popularity_group"] == group, "item"
        ]
    )
    for group in _groups
}

_group_columns = {
    group: {_item_column[item] for item in items}
    for group, items in ITEM_SETS_3_4.items()
}

# Relevant item columns for each user and popularity group.
_group_truth = {
    group: {
        str(user): {
            _item_column[item]
            for item in GT_TEST[str(user)]
            if item in ITEM_SETS_3_4[group]
        }
        for user in USERS_TEST
    }
    for group in _groups
}

_item_rows = []

for system, lists in LISTS_3_4.items():
    shown = set(lists.ravel())

    for group in _groups:
        columns = _group_columns[group]
        recalls = []
        total_hits = 0
        relevant_items = 0

        for user, recommended in zip(USERS_TEST, lists):
            relevant = _group_truth[group][str(user)]
            if not relevant:
                continue

            hits = len(set(recommended) & relevant)
            recalls.append(hits / len(relevant))
            total_hits += hits
            relevant_items += len(relevant)

        _item_rows.append({
            "system": system,
            "group": group,
            "items": len(columns),
            "users_with_relevant": len(recalls),
            "relevant_items": relevant_items,
            "hits": total_hits,
            "recall": float(np.mean(recalls)) if recalls else np.nan,
            "exposure": float(np.isin(lists, list(columns)).mean()),
            "coverage": len(shown & columns) / len(columns),
        })

ITEM_RESULTS_3_4 = pd.DataFrame(_item_rows).merge(
    SYSTEMS_3_4, on="system", validate="many_to_one"
)

_metrics = ["recall", "exposure", "coverage", "hits"]
_base = ITEM_RESULTS_3_4[
    ITEM_RESULTS_3_4["method"] == "base"
][["system", "group", *_metrics]].rename(
    columns={
        "system": "base",
        **{metric: f"base_{metric}" for metric in _metrics},
    }
)

ITEM_RESULTS_3_4 = ITEM_RESULTS_3_4.merge(
    _base,
    on=["base", "group"],
    how="left",
    validate="many_to_one",
)

for metric in _metrics:
    assert ITEM_RESULTS_3_4[f"base_{metric}"].notna().all()
    ITEM_RESULTS_3_4[f"delta_{metric}"] = (
        ITEM_RESULTS_3_4[metric]
        - ITEM_RESULTS_3_4[f"base_{metric}"]
    )

assert len(ITEM_RESULTS_3_4) == 183
assert np.allclose(
    ITEM_RESULTS_3_4.groupby("system")["exposure"].sum(), 1.0
)

print(f"Item-group results: {len(ITEM_RESULTS_3_4)} rows")

# Show the intervention aimed directly at popularity-group exposure.
_columns = [
    "family", "model", "placement", "group",
    "recall", "delta_recall",
    "exposure", "delta_exposure",
    "coverage", "delta_coverage",
]
_item_fairness = ITEM_RESULTS_3_4[
    ITEM_RESULTS_3_4["method"] == "item_fair"
]

with pd.option_context("display.float_format", "{:.4f}".format):
    print("\nItem fairness: performance by popularity group")
    print(_item_fairness[_columns].to_string(index=False))

ITEM_RESULTS_3_4.to_csv(
    RUNS / "tables" / "table_3_4_items.csv", index=False
)

In [ ]:
# =============================================================================
# 3.4.4  NDCG changes across profile-length groups
# NDCG changes by profile group; all panels share the same colour scale.
# =============================================================================
_user_results = pd.read_csv(
    RUNS / "tables" / "table_3_4_users.csv"
)

_profile = _user_results[
    (_user_results["group_type"] == "profile")
    & (_user_results["method"] != "base")
].copy()

_profile["display"] = np.where(
    _profile["family"] == "individual",
    _profile["model"],
    _profile["model"] + " / " + _profile["placement"],
)

_order = ["EASE", "SLIMElastic", "NGCF", "NeuMF", "BPR"]
_order += [
    f"{model} / {placement}"
    for model in ["Weighted", "Switching", "Cascade", "Mixed"]
    for placement in ["pre", "post"]
]

_methods = [
    ("mmr", "Diversification"),
    ("calibration", "Calibration"),
    ("item_fair", "Item-side fairness"),
    ("user_fair", "User-side fairness"),
]

_limit = max(float(_profile["delta_ndcg"].abs().max()), 1e-12)
fig, axes = plt.subplots(1, 4, figsize=(15, 7), sharey=True)

for ax, (method, title) in zip(axes, _methods):
    table = (
        _profile[_profile["method"] == method]
        .pivot(index="display", columns="group", values="delta_ndcg")
        .reindex(index=_order, columns=["low", "mid", "high"])
    )
    assert table.notna().all().all()

    image = ax.imshow(
        table.to_numpy(), cmap="RdBu",
        vmin=-_limit, vmax=_limit, aspect="auto",
    )

    ax.set_title(title)
    ax.set_xticks(range(3), ["Low", "Mid", "High"])
    ax.set_yticks(range(len(_order)), _order)

    for row in range(len(_order)):
        for col in range(3):
            value = table.iloc[row, col]
            ax.text(
                col, row, f"{value:+.3f}",
                ha="center", va="center", fontsize=8,
                color="white" if abs(value) > 0.6 * _limit else "black",
            )

fig.subplots_adjust(left=0.16, right=0.90, wspace=0.25, top=0.90)
_colorbar = fig.add_axes([0.92, 0.15, 0.015, 0.70])
fig.colorbar(
    image, cax=_colorbar, label="NDCG@10 change from own base"
)
fig.savefig(
    RUNS / "tables" / "fig_task34_users.png",
    dpi=150, bbox_inches="tight",
)
plt.show()

In [ ]:
# =============================================================================
# 3.4.5  Item fairness: exposure and recall changes
# Item fairness: exposure and recall changes use separate colour scales.
# =============================================================================
_item_results = pd.read_csv(
    RUNS / "tables" / "table_3_4_items.csv"
)

_item_fairness = _item_results[
    _item_results["method"] == "item_fair"
].copy()

_item_fairness["display"] = np.where(
    _item_fairness["family"] == "individual",
    _item_fairness["model"],
    _item_fairness["model"] + " / " + _item_fairness["placement"],
)

_order = ["EASE", "SLIMElastic", "NGCF", "NeuMF", "BPR"]
_order += [
    f"{model} / {placement}"
    for model in ["Weighted", "Switching", "Cascade", "Mixed"]
    for placement in ["pre", "post"]
]

fig, axes = plt.subplots(
    1, 2, figsize=(11, 7), sharey=True,
    constrained_layout=True,
)

for ax, metric, title in zip(
    axes,
    ["delta_exposure", "delta_recall"],
    ["Exposure share change", "Group recall change"],
):
    table = (
        _item_fairness.pivot(
            index="display", columns="group", values=metric
        )
        .reindex(index=_order, columns=["head", "mid", "tail"])
    )
    assert table.notna().all().all()

    limit = max(float(np.abs(table.to_numpy()).max()), 1e-12)
    image = ax.imshow(
        table.to_numpy(), cmap="RdBu",
        vmin=-limit, vmax=limit, aspect="auto",
    )

    ax.set_title(title)
    ax.set_xticks(range(3), ["Head", "Mid", "Tail"])
    ax.set_yticks(range(len(_order)), _order)

    for row in range(len(_order)):
        for col in range(3):
            value = table.iloc[row, col]
            ax.text(
                col, row, f"{value:+.3f}",
                ha="center", va="center", fontsize=9,
                color="white" if abs(value) > 0.6 * limit else "black",
            )

    fig.colorbar(image, ax=ax, shrink=0.75)
    
fig.savefig(
    RUNS / "tables" / "fig_task34_items.png",
    dpi=150, bbox_inches="tight",
)
plt.show()

In [ ]:
# =============================================================================
# 3.4.6  Target improvements across profile-length groups
# =============================================================================
_user_results = pd.read_csv(
    RUNS / "tables" / "table_3_4_users.csv"
)

_profile = _user_results[
    _user_results["group_type"] == "profile"
].copy()

_profile["display"] = np.where(
    _profile["family"] == "individual",
    _profile["model"],
    _profile["model"] + " / " + _profile["placement"],
)

for method, metric, direction, title in [
    ("mmr", "delta_ild", 1, "MMR: ILD increase"),
    ("calibration", "delta_cal_kl", -1, "Calibration: genre KL reduction"),
]:
    rows = _profile[_profile["method"] == method].copy()
    rows["improvement"] = direction * rows[metric]

    table = rows.pivot(
        index="display", columns="group", values="improvement"
    ).reindex(columns=["low", "mid", "high"])

    print(f"\n{title}")
    print(table.round(4).to_string())

### Reading Task 3.4

Reranking benefits and accuracy costs are unevenly distributed.

MMR improves ILD across all profile-length groups where applied,
but accuracy losses differ. For EASE with MMR, NDCG falls by 0.009 for low-activity users and 0.024 for high-activity users, despite similar ILD gains. Overall averages therefore hide differences in who bears the cost. Calibration reduces genre KL across all profile groups where applied.

Under item-side fairness, exposure shifts from head items towards
mid and tail items in the non-fallback settings. Tail recall also
increases, so the change is not limited to displaying more
unpopular items. Head recall falls in exchange. Cascade's post-hybrid fallback
leaves these measures unchanged.

User-side fairness preserves women's recommendations and lowers
men's NDCG. The smaller gender gap therefore reflects reduced
advantage rather than improved accuracy for women.

These are descriptive group comparisons. Profile length affects
held-out set size, and gender groups differ in composition;
the results do not establish causal effects or statistical
significance.